# 06c · Phase 3: Faster R-CNN Shard

Trains 6 loss configs × 3 splits = **18 runs** with Faster R-CNN R50-FPN via MMDetection.  
Est. runtime: **~4 h** on Kaggle T4 (includes ~15 min MMDetection install).

Part of the Phase 3 cross-architecture study. Run in parallel with 06a and 06b.

## Section 1 · Environment

In [ ]:
# Environment detection
import os, sys
from pathlib import Path

ON_KAGGLE = os.path.exists('/kaggle/working')
ON_COLAB  = 'google.colab' in sys.modules or os.path.exists('/content')

if ON_KAGGLE:
    ROOT = Path('/kaggle/working')
    print('Runtime: Kaggle')
elif ON_COLAB:
    ROOT = Path('/content')
    print('Runtime: Colab')
else:
    ROOT = Path.cwd()
    print(f'Runtime: local ({ROOT})')

gpu = os.popen('nvidia-smi --query-gpu=name --format=csv,noheader').read().strip()
print(f'GPU: {gpu or "none detected"}')


In [ ]:
# Install core dependencies
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'ultralytics==8.4.9', 'wandb==0.24.1', 'pycocotools'], check=True)
print('Core deps installed.')


In [ ]:
# Install MMDetection (Faster R-CNN)
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'openmim'], check=True)
subprocess.run(['mim', 'install', '-q', 'mmengine', 'mmcv>=2.0',
                'mmdet'], check=True)
print('MMDetection installed.')


In [ ]:
# Clone / update repo
import os, sys
from pathlib import Path

REPO_PATH = ROOT / 'amorphous-yolo'
if not (REPO_PATH / '.git').exists():
    os.system(f'git clone https://github.com/nipun-taneja/amorphous-yolo.git {REPO_PATH}')
    print('Cloned.')
else:
    os.system(f'git -C {REPO_PATH} pull --ff-only')
    print('Repo up to date.')
if str(REPO_PATH) not in sys.path:
    sys.path.insert(0, str(REPO_PATH))


In [ ]:
# WandB (optional)
import os, wandb
WANDB_PROJECT = 'amorphous-yolo-phase3'
if ON_KAGGLE:
    try:
        from kaggle_secrets import UserSecretsClient
        _k = UserSecretsClient().get_secret('WANDB_API_KEY')
        if _k: os.environ['WANDB_API_KEY'] = _k
    except Exception: pass
elif ON_COLAB:
    try:
        from google.colab import userdata
        _k = userdata.get('WANDB_API_KEY')
        if _k: os.environ['WANDB_API_KEY'] = _k
    except Exception: pass
if os.environ.get('WANDB_API_KEY'):
    wandb.login(key=os.environ['WANDB_API_KEY'], relogin=True)
    print(f'WandB active → {WANDB_PROJECT}')
else:
    os.environ['WANDB_MODE'] = 'disabled'
    print('WandB disabled.')


## Section 2 · Constants & Dataset

In [ ]:
# Phase 3 constants & paths
import json as _json, math, time
from pathlib import Path
from datetime import datetime

PROJECT_DIR   = ROOT / 'amorphous-yolo'
DATASET_ROOT  = PROJECT_DIR / 'data' / 'kvasir_seg'
EXPERIMENTS   = ROOT / 'experiments_phase3'
ANALYSIS_DIR  = EXPERIMENTS / 'analysis'
METRICS_DIR   = EXPERIMENTS / 'metrics'
MANIFEST_PATH = EXPERIMENTS / 'manifest.json'
for _d in [EXPERIMENTS, ANALYSIS_DIR, METRICS_DIR]:
    _d.mkdir(parents=True, exist_ok=True)

EPOCHS = 20
IMGSZ  = 640
SEEDS  = [42]
DEVICE = 0

SPLIT_CONFIGS = {
    'clean': PROJECT_DIR / 'data' / 'kvasir_seg.yaml',
    'low':   PROJECT_DIR / 'data' / 'kvasir_seg_low.yaml',
    'high':  PROJECT_DIR / 'data' / 'kvasir_seg_high.yaml',
}

# Drive persistence (Colab only)
DRIVE_AVAILABLE = False
if ON_COLAB:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        DRIVE_EXPERIMENTS = Path('/content/drive/MyDrive/experiments_phase3')
        DRIVE_EXPERIMENTS.mkdir(parents=True, exist_ok=True)
        DRIVE_AVAILABLE = True
        print(f'Drive mounted: {DRIVE_EXPERIMENTS}')
    except Exception as e:
        print(f'Drive not available: {e}')
print(f'EXPERIMENTS = {EXPERIMENTS}')


In [ ]:
# Download / verify Kvasir-SEG dataset
import os
VALID_DIR  = DATASET_ROOT / 'valid' / 'images'
LOW_DIR    = DATASET_ROOT / 'valid_low' / 'images'
HIGH_DIR   = DATASET_ROOT / 'valid_high' / 'images'

if not VALID_DIR.exists():
    print('Downloading Kvasir-SEG...')
    os.system(f'cd {PROJECT_DIR} && python scripts/prepare_kvasir.py')
n_imgs = len(list(VALID_DIR.glob('*.jpg'))) if VALID_DIR.exists() else 0
print(f'Val images: {n_imgs}  (expected 200)')


## Section 3 · Loss Configs

In [ ]:
# Import loss classes
from src.losses import (
    CIoULoss, EIoULoss, ECIoULoss, AEIoULoss,
)
import ultralytics.utils.loss as _ul
_ORIG_BBOX_FWD = _ul.BboxLoss.forward

def patch_loss(loss_fn):
    import torch
    def _fwd(self, pred_dist, pred_bboxes, anchor_points, target_bboxes,
             target_scores, target_scores_sum, fg_mask):
        loss_iou = loss_fn(
            pred_bboxes[fg_mask], target_bboxes[fg_mask]
        ) * target_scores[fg_mask].sum() / target_scores_sum
        loss_dfl = torch.tensor(0.0, device=pred_dist.device)
        if self.use_dfl:
            loss_dfl = self.dfl_loss(
                pred_dist[fg_mask].view(-1, self.reg_max + 1),
                target_bboxes[fg_mask],
                anchor_points,
            ) * target_scores[fg_mask].sum() / target_scores_sum
        return loss_iou, loss_dfl
    _ul.BboxLoss.forward = _fwd

def restore_loss():
    _ul.BboxLoss.forward = _ORIG_BBOX_FWD

# Auto-select best AEIoU rigidities from Phase 2 (fallback to defaults)
import json as _json
_p2_metrics = Path('/kaggle/working/amorphous-yolo/experiments_kvasir/metrics_all_losses.json')
if not _p2_metrics.exists():
    _p2_metrics = Path('/content/amorphous-yolo/experiments_kvasir/metrics_all_losses.json')
try:
    _data = _json.loads(_p2_metrics.read_text())
    _aeiou = [(v.get('map50_95', 0), v['loss']) for v in _data.values()
              if v.get('loss', '').startswith('aeiou')]
    _top3 = sorted(_aeiou, reverse=True)[:3]
    AEIOU_RIGIDITIES = [float(n.split('r')[-1].replace('p', '.')) for _, n in _top3]
    print(f'Best AEIoU rigidities from Phase 2: {AEIOU_RIGIDITIES}')
except Exception:
    AEIOU_RIGIDITIES = [0.1, 0.3, 0.5]
    print(f'Phase 2 metrics not found. Using default rigidities: {AEIOU_RIGIDITIES}')

LOSS_CONFIGS = [
    ('ciou',  CIoULoss()),
    ('eiou',  EIoULoss()),
    ('eciou', ECIoULoss()),
] + [(f'aeiou_r{str(r).replace(".", "p")}', AEIoULoss(rigidity=r)) for r in AEIOU_RIGIDITIES]
print(f'Loss configs: {[n for n,_ in LOSS_CONFIGS]}')


## Section 4 · COCO Annotations for MMDetection

In [ ]:
# Convert YOLO labels → COCO annotations for MMDetection DataLoader
import json as _json, cv2

def yolo_to_coco(img_dir, lbl_dir, out_path, split_tag):
    if Path(out_path).exists(): return
    Path(out_path).parent.mkdir(parents=True, exist_ok=True)
    imgs, anns, ann_id = [], [], 1
    for p in sorted(Path(img_dir).glob('*.jpg')):
        im = cv2.imread(str(p)); H, W = im.shape[:2]
        img_id = int(p.stem) if p.stem.isdigit() else hash(p.stem) % 10**6
        imgs.append({'id': img_id, 'file_name': str(p), 'width': W, 'height': H})
        lbl = Path(lbl_dir) / f'{p.stem}.txt'
        if not lbl.exists(): continue
        for line in lbl.read_text().strip().splitlines():
            _, cx, cy, bw, bh = map(float, line.split())
            x1, y1 = (cx-bw/2)*W, (cy-bh/2)*H
            anns.append({'id': ann_id, 'image_id': img_id, 'category_id': 1,
                          'bbox': [x1, y1, bw*W, bh*H],
                          'area': bw*W*bh*H, 'iscrowd': 0})
            ann_id += 1
    d = {'images': imgs, 'annotations': anns,
         'categories': [{'id': 1, 'name': 'polyp'}]}
    Path(out_path).write_text(_json.dumps(d))
    print(f'COCO JSON ({split_tag}): {len(imgs)} imgs, {len(anns)} anns → {out_path}')

COCO_DIR = DATASET_ROOT / 'coco_annotations'
yolo_to_coco(DATASET_ROOT/'train'/'images', DATASET_ROOT/'train'/'labels',
             COCO_DIR/'instances_train.json', 'train')
yolo_to_coco(DATASET_ROOT/'valid'/'images', DATASET_ROOT/'valid'/'labels',
             COCO_DIR/'instances_val.json', 'val/clean')
yolo_to_coco(DATASET_ROOT/'valid_low'/'images', DATASET_ROOT/'valid_low'/'labels',
             COCO_DIR/'instances_val_low.json', 'val/low')
yolo_to_coco(DATASET_ROOT/'valid_high'/'images', DATASET_ROOT/'valid_high'/'labels',
             COCO_DIR/'instances_val_high.json', 'val/high')


In [ ]:
# Build COCO GT JSON from YOLO val labels (idempotent)
import cv2, json as _json

def build_coco_gt_json(img_dir, lbl_dir, out_path):
    if Path(out_path).exists(): return
    images, anns, ann_id = [], [], 1
    for p in sorted(Path(img_dir).glob('*.jpg')):
        img = cv2.imread(str(p))
        H, W = img.shape[:2]
        img_id = int(p.stem) if p.stem.isdigit() else hash(p.stem) % 10**6
        images.append({'id': img_id, 'file_name': p.name, 'width': W, 'height': H})
        lbl = Path(lbl_dir) / f'{p.stem}.txt'
        if not lbl.exists(): continue
        for line in lbl.read_text().strip().splitlines():
            _, cx, cy, bw, bh = map(float, line.split())
            x1, y1 = (cx-bw/2)*W, (cy-bh/2)*H
            anns.append({'id': ann_id, 'image_id': img_id, 'category_id': 1,
                          'bbox': [x1, y1, bw*W, bh*H],
                          'area': bw*W*bh*H, 'iscrowd': 0})
            ann_id += 1
    coco = {'images': images, 'annotations': anns,
            'categories': [{'id': 1, 'name': 'polyp'}]}
    Path(out_path).parent.mkdir(parents=True, exist_ok=True)
    Path(out_path).write_text(_json.dumps(coco))
    print(f'COCO GT JSON: {out_path} ({len(images)} imgs, {len(anns)} anns)')

COCO_GT_JSON = DATASET_ROOT / 'valid' / 'coco_gt.json'
build_coco_gt_json(
    DATASET_ROOT / 'valid' / 'images',
    DATASET_ROOT / 'valid' / 'labels',
    COCO_GT_JSON,
)


## Section 5 · MMDet Loss Registration

In [ ]:
# Register custom losses with MMDetection model registry
from mmdet.registry import MODELS
from src.losses import CIoULoss, EIoULoss, ECIoULoss, AEIoULoss
import torch

def _make_mmdet_loss(loss_cls, **kwargs):
    """Wrap a src.losses class to the MMDet loss API."""
    _inst = loss_cls(**kwargs)
    @MODELS.register_module(name=f'{loss_cls.__name__}MMDet_{id(_inst)}')
    class _Wrapper(torch.nn.Module):
        def __init__(self):
            super().__init__()
            self.loss_fn = _inst
            self.loss_weight = 1.0
        def forward(self, pred, target, weight=None, avg_factor=None, **kw):
            return self.loss_fn(pred, target)
    return _Wrapper.__name__, _Wrapper

# Build MMDet loss registry entries for each loss config
MMDET_LOSS_MAP = {}  # loss_name → registered MMDet class name
for loss_name, loss_fn_instance in LOSS_CONFIGS:
    reg_name, cls = _make_mmdet_loss(type(loss_fn_instance),
                                     **({'rigidity': loss_fn_instance.rigidity}
                                        if hasattr(loss_fn_instance, 'rigidity') else {}))
    MMDET_LOSS_MAP[loss_name] = reg_name
print('Registered MMDet losses:', list(MMDET_LOSS_MAP.keys()))


## Section 6 · Faster R-CNN Training Functions

In [ ]:
# build_frcnn_config() + run_frcnn()
from mmengine.config import Config
from mmengine.runner import Runner
import json as _json, time, shutil
from datetime import datetime

FRCNN_CFGS_DIR = EXPERIMENTS / 'frcnn_configs'
FRCNN_CFGS_DIR.mkdir(exist_ok=True)

def build_frcnn_config(loss_name, split_name, run_dir, seed, epochs):
    cfg = Config.fromfile('configs/faster_rcnn/faster-rcnn_r50_fpn_1x_coco.py')
    loss_dict = {'type': MMDET_LOSS_MAP[loss_name], 'loss_weight': 1.0}
    cfg.data_root = str(DATASET_ROOT)
    cfg.metainfo = {'classes': ('polyp',), 'palette': [(220, 20, 60)]}
    ann_tag = {'clean': 'val', 'low': 'val_low', 'high': 'val_high'}[split_name]
    val_img = str({'clean': DATASET_ROOT/'valid'/'images',
                   'low':   DATASET_ROOT/'valid_low'/'images',
                   'high':  DATASET_ROOT/'valid_high'/'images'}[split_name])
    for d in [cfg.train_dataloader.dataset, cfg.val_dataloader.dataset,
              cfg.test_dataloader.dataset]:
        d.metainfo = cfg.metainfo
    cfg.train_dataloader.dataset.ann_file = str(DATASET_ROOT/'coco_annotations'/'instances_train.json')
    cfg.train_dataloader.dataset.data_prefix = dict(img=str(DATASET_ROOT/'train'/'images'))
    cfg.val_dataloader.dataset.ann_file   = str(DATASET_ROOT/f'coco_annotations/instances_{ann_tag}.json')
    cfg.val_dataloader.dataset.data_prefix = dict(img=val_img)
    cfg.test_dataloader.dataset.ann_file  = cfg.val_dataloader.dataset.ann_file
    cfg.test_dataloader.dataset.data_prefix = cfg.val_dataloader.dataset.data_prefix
    cfg.val_evaluator.ann_file  = cfg.val_dataloader.dataset.ann_file
    cfg.test_evaluator.ann_file = cfg.val_dataloader.dataset.ann_file
    for head in [cfg.model.rpn_head, cfg.model.roi_head.bbox_head]:
        if hasattr(head, 'loss_bbox'): head.loss_bbox = loss_dict
    cfg.model.roi_head.bbox_head.num_classes = 1
    cfg.train_cfg.max_epochs = epochs
    cfg.default_hooks.checkpoint.interval = epochs
    cfg.work_dir = str(run_dir)
    cfg.seed = seed
    cfg_path = FRCNN_CFGS_DIR / f'{run_dir.name}.py'
    cfg.dump(str(cfg_path))
    return cfg_path

def run_frcnn(loss_name, split_name, seed=42, epochs=None):
    epochs = epochs or EPOCHS
    run_name = f'phase3_frcnn_{loss_name}_{split_name}_s{seed}_e{epochs}'
    run_dir  = EXPERIMENTS / run_name
    result_json = run_dir / 'metric.json'
    if result_json.exists():
        print(f'[SKIP] {run_name}'); return run_dir
    run_dir.mkdir(exist_ok=True)
    print(f'\n{"="*65}\n[START] {run_name}\n{"="*65}')
    cfg_path = build_frcnn_config(loss_name, split_name, run_dir, seed, epochs)
    t0 = time.time()
    try:
        runner = Runner.from_cfg(Config.fromfile(str(cfg_path)))
        runner.train()
        # Save metrics
        metrics = runner.val()
        result_json.write_text(_json.dumps(metrics, indent=2))
        print(f'[DONE] {run_name}  ({time.time()-t0:.0f}s)')
        if DRIVE_AVAILABLE:
            try: shutil.copytree(str(run_dir), str(DRIVE_EXPERIMENTS/run_name), dirs_exist_ok=True)
            except: pass
    except Exception as e:
        print(f'  [ERROR] {run_name}: {e}'); raise
    return run_dir
print('run_frcnn() ready.')


## Section 7 · Training Loop

In [ ]:
# Faster R-CNN — 18 runs (6 losses × 3 splits)
total, done, skipped, failed = 0, 0, 0, 0
for loss_name, _ in LOSS_CONFIGS:
    for split_name in SPLIT_CONFIGS:
        total += 1
        try:
            result = run_frcnn(loss_name, split_name)
            if result: done += 1
            else: skipped += 1
        except Exception as e:
            print(f'  [ERROR] {loss_name}/{split_name}: {e}'); failed += 1
print(f'\nFaster R-CNN: {done} done, {skipped} skipped, {failed} failed / {total} total')


## Section 8 · Save & Output

In [ ]:

# Save Faster R-CNN summary CSV
import pandas as pd, json as _json

ARCH_PREFIX = 'frcnn'
rows = []
for run_dir in sorted(EXPERIMENTS.glob('phase3_frcnn_*')):
    mj = run_dir / 'metric.json'
    if not mj.exists(): continue
    m = _json.loads(mj.read_text())
    run_name = run_dir.name
    parts = run_name.split('_')
    rows.append({
        'run': run_name, 'arch': 'frcnn',
        'loss':  parts[3] if len(parts) > 3 else '',
        'split': parts[4] if len(parts) > 4 else '',
        'map50':    m.get('coco/bbox_mAP_50', 0),
        'map50_95': m.get('coco/bbox_mAP', 0),
    })

summary_csv = EXPERIMENTS / 'summary_frcnn.csv'
pd.DataFrame(rows).to_csv(summary_csv, index=False)
print(f'Summary saved: {summary_csv}  ({len(rows)} runs)')
print(pd.DataFrame(rows).to_string(index=False))


In [ ]:

# ── OUTPUT INSTRUCTIONS ──────────────────────────────────────────────────────
# After this notebook finishes:
#
# On Kaggle:
#   1. Go to this notebook's Output tab
#   2. Click "+ New Dataset" → name it  phase3-{arch}-results
#   3. In 06d_analysis.ipynb, add this dataset as an Input
#      Path will be: /kaggle/input/phase3-{arch}-results/
#
# On Colab:
#   All outputs already synced to Drive → experiments_phase3/
#   Run 06d directly without any extra steps.
print('Shard complete. See comment above for output publishing instructions.')
